# Fine-tuning de BETO para sentimiento en español

Cuaderno de la lección 11 del bloque 5 del curso **Deep Learning para NLP**
(`gustavoai.dev/cursos/dl-nlp/fine-tuning-colab`).

Aquí se hace lo que el navegador no puede: partir de un modelo preentrenado de verdad
—BETO, el BERT en español de la Universidad de Chile— y seguir el descenso de gradiente
unas pocas vueltas sobre unos miles de reseñas etiquetadas.

El bucle de entrenamiento está escrito a mano, con su `loss.backward()` a la vista, en vez
de con la clase `Trainer` de la biblioteca: es el mismo bucle de la lección 6 del bloque 2,
y la única línea que hace algo que no escribiste tú es precisamente ésa.

**Antes de ejecutar nada:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.
La celda de más abajo comprueba si te ha tocado una.

## 1. Versiones

Fijadas a propósito. Unos pesos preentrenados y una versión concreta de la biblioteca son
las dos mitades de un resultado reproducible; dejar la segunda al aire es la manera más
silenciosa de que este cuaderno deje de funcionar dentro de un año.

`torch` no se fija: viene con Colab, compilado contra el CUDA de la máquina, y reinstalarlo
es la forma más rápida de quedarse sin GPU. Lo que sí se hace es dejar escrito con cuál se
ejecutó.

In [ ]:
!pip install -q transformers==4.57.6 datasets==4.8.5 accelerate==1.14.0

In [ ]:
import torch, transformers, datasets

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)

## 2. ¿Hay GPU?

Si la hay, las tres vueltas de más abajo tardan unos minutos. Si no la hay el cuaderno
corre igual, sobre CPU, y entonces la cuenta de la lección se te viene encima: son del
orden de 7 × 10¹⁴ multiplicaciones.

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("memoria: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("Sin GPU. Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU.")
    print("Puedes seguir, pero baja N_ENT en la celda de datos o te vas a quedar aquí.")

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

## 3. El modelo y su tokenizador

El tokenizador viene **con** el modelo y no lo eliges tú: sus entradas son exactamente las
filas de la tabla de embeddings preentrenada, así que cambiarlo dejaría esas filas sin dueño.

`num_labels=2` es lo único que se añade encima de la pila: una capa de salida de dos unidades
sobre los 768 números de la primera posición. Ésos son los pesos que salen del generador de
números aleatorios, y el aviso que imprime la biblioteca dice exactamente eso.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"

tok = AutoTokenizer.from_pretrained(CHECKPOINT)
modelo = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2)
modelo.to(dispositivo)
print()

### La cuenta de la lección, comprobada

Los números que la lección deriva a mano salen de aquí. Los dos primeros son la fórmula de
la lección 8 del bloque 5 evaluada en la configuración de este modelo; el tercero es lo que
de verdad hay que mover.

In [ ]:
cfg = modelo.config
por_bloque = 4 * cfg.hidden_size**2 + 2 * cfg.hidden_size * cfg.intermediate_size
bloques = cfg.num_hidden_layers * por_bloque
tabla = cfg.vocab_size * cfg.hidden_size
cabeza = 2 * cfg.hidden_size + 2

print("N=%d  d_model=%d  h=%d  d_ff=%d  |V|=%d" % (
    cfg.num_hidden_layers, cfg.hidden_size,
    cfg.num_attention_heads, cfg.intermediate_size, cfg.vocab_size))
print("por bloque      : %12s" % f"{por_bloque:,}")
print("los %2d bloques  : %12s" % (cfg.num_hidden_layers, f"{bloques:,}"))
print("tabla embeddings: %12s" % f"{tabla:,}")
print("capa de salida  : %12s" % f"{cabeza:,}")
print("total real      : %12s" % f"{sum(p.numel() for p in modelo.parameters()):,}")
print("en memoria      : %.0f MB solo los pesos" % (
    sum(p.numel() for p in modelo.parameters()) * 4 / 1e6))

## 4. Las reseñas

Reseñas de producto en español, con su puntuación de una a cinco estrellas (en el archivo, la
etiqueta va de 0 a 4). Se tiran las de
tres —son las que no dicen ni que sí ni que no— y las demás se parten en dos: negativa (0)
para una y dos estrellas, positiva (1) para cuatro y cinco.

`N_ENT` y `N_PRU` son deliberadamente pequeños: unos miles de ejemplos etiquetados es todo
lo que el *fine-tuning* pide, y es la mitad de la tesis de la lección.

Si este conjunto desapareciera del Hub, cualquier CSV con una columna de texto y una de
etiqueta binaria sirve: lo único que este cuaderno necesita de él son las columnas `text` y
`labels`.

In [ ]:
from datasets import load_dataset

N_ENT, N_PRU, T_MAX = 3000, 1000, 128

BASE = "https://huggingface.co/datasets/mteb/amazon_reviews_multi/resolve/refs%2Fconvert%2Fparquet/es"
bruto = load_dataset("parquet", data_files={
    "train": f"{BASE}/train/0000.parquet",
    "test":  f"{BASE}/test/0000.parquet",
})

def util(ej):
    return ej["label"] != 2          # fuera las de tres estrellas (la etiqueta 2 de 0..4)

def binaria(ej):
    return {"labels": 1 if ej["label"] > 2 else 0}

ent = bruto["train"].filter(util).shuffle(seed=0).select(range(N_ENT)).map(binaria)
pru = bruto["test"].filter(util).shuffle(seed=0).select(range(N_PRU)).map(binaria)

print("D_ent   :", len(ent), " positivas:", sum(ent["labels"]))
print("D_prueba:", len(pru), " positivas:", sum(pru["labels"]))
print()
print(ent[0]["text"][:300])
print("-> etiqueta", ent[0]["labels"])

### Trocear y truncar

Cada reseña pasa por el tokenizador de subpalabras del modelo y se trunca a `T_MAX = 128`
posiciones. Merece la pena mirar la primera: dónde parte las palabras que no están enteras
en el vocabulario es exactamente lo de la lección 2 del bloque 1.

In [ ]:
def trocea(lote):
    return tok(lote["text"], truncation=True, max_length=T_MAX, padding="max_length")

ent_tok = ent.map(trocea, batched=True)
pru_tok = pru.map(trocea, batched=True)

columnas = ["input_ids", "attention_mask", "labels"]
ent_tok.set_format("torch", columns=columnas)
pru_tok.set_format("torch", columns=columnas)

print(tok.tokenize("La batería dura muchísimo y el envío fue rapidísimo."))

## 5. Tres vueltas

Éste es el bucle de la lección 6 del bloque 2, con dos diferencias y ninguna más:
`theta_0` sale del preentrenamiento en vez del generador, y la tasa de aprendizaje es unas
cincuenta veces menor que la de aquel bloque —`2e-5` contra `1e-3`— porque un paso del
tamaño de allí sacaría a `theta` del sitio que costó semanas de máquina.

`salida.loss.backward()` es la recurrencia de la lección 8 del bloque 2, corriendo sobre un
grafo que la biblioteca construyó sola durante el forward pass.

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

ETA, VUELTAS, LOTE = 2e-5, 3, 16

cargador_ent = DataLoader(ent_tok, batch_size=LOTE, shuffle=True)
cargador_pru = DataLoader(pru_tok, batch_size=64)
opt = AdamW(modelo.parameters(), lr=ETA)

for vuelta in range(1, VUELTAS + 1):
    modelo.train()
    suma, vistos = 0.0, 0
    for lote in cargador_ent:
        lote = {k: v.to(dispositivo) for k, v in lote.items()}
        salida = modelo(**lote)
        salida.loss.backward()
        opt.step()
        opt.zero_grad()
        n = lote["labels"].shape[0]
        suma += salida.loss.item() * n
        vistos += n
    print("vuelta %d   L = %.4f" % (vuelta, suma / vistos))

## 6. Sobre las que no ha leído

La tasa de acierto sobre `D_prueba`. Lo que hay que mirar no es tanto cuánto vale como qué
ha hecho falta para conseguirlo: tres vueltas, unos miles de ejemplos y unos minutos.
Compáralo con el 88.3 % que el proyecto del bloque 2 sacó entrenando desde cero, y con las
semanas de preentrenamiento que hay debajo de este número.

In [ ]:
modelo.eval()
aciertos = 0
with torch.no_grad():
    for lote in cargador_pru:
        lote = {k: v.to(dispositivo) for k, v in lote.items()}
        pred = modelo(**lote).logits.argmax(dim=-1)
        aciertos += (pred == lote["labels"]).sum().item()

print("acierto(D_prueba) = %.3f  (%d de %d)" % (
    aciertos / len(pru_tok), aciertos, len(pru_tok)))

## 7. Tus propias frases

Cámbialas por lo que quieras. Prueba en particular una frase positiva y la misma con un
*no* delante: eso es lo que la bolsa de palabras del bloque 2 no sabía dónde colocar.

In [ ]:
frases = [
    "La batería dura muchísimo y el envío llegó antes de tiempo.",
    "No me ha gustado nada: se rompió a la semana.",
    "Esperaba bastante más por este precio.",
    "No está mal del todo, la verdad.",
]

modelo.eval()
with torch.no_grad():
    x = tok(frases, truncation=True, max_length=T_MAX, padding=True, return_tensors="pt")
    x = {k: v.to(dispositivo) for k, v in x.items()}
    p = modelo(**x).logits.softmax(dim=-1)

for frase, fila in zip(frases, p):
    print("%.3f positiva  |  %s" % (fila[1].item(), frase))

## 8. Guardar

Los pesos ajustados, con su tokenizador al lado. Sin el tokenizador no valen nada: es el que
sabe qué entrada del vocabulario le corresponde a cada fila de la tabla.

In [ ]:
modelo.save_pretrained("beto-sentimiento")
tok.save_pretrained("beto-sentimiento")
!du -sh beto-sentimiento

---

### Qué tocar a partir de aquí

- **`ETA`**. Súbelo a `1e-3`, el del bloque 2, y vuelve a correr desde la celda del modelo.
  Mira la pérdida y mira después el acierto: es la segunda pregunta de la lección, medida.
- **`N_ENT`**. Bájalo a 300 y mira cuánto se pierde. Es menos de lo que parece, y ésa es la
  razón entera de que el preentrenamiento importe.
- **`CHECKPOINT`**. Cualquier otro modelo en español del Hub entra por la misma puerta; el
  resto del cuaderno no cambia una línea.